### All Imports and Libraries

In [1]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.poisson_ci_detector import ImprovedPoissonConcentrationML
from src.features import (
    infer_protocol_column,
    compute_jitter_iat_features,
    compute_source_flow_concentration,
    generate_concentration_weighted_ci
)
from src.evaluation import (
    poisson_diagnostic,
    evaluate_at_fprs
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, mean_squared_error
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split 

import warnings
warnings.filterwarnings("ignore", message="Overdispersion detected in >25% of features.")

### Global Storage for all Attacks

# PHASE 1a IMPLEMENTATION

### Load and Inspect Benign

In [2]:
# ============================================================
# STEP 1: LOAD BENIGN DATA
# ============================================================

BASE_PATH = "../data"

benign_path = os.path.join(BASE_PATH, "BenignTraffic1.csv")
df_benign = pd.read_csv(benign_path, low_memory=False)

# ------------------------------------------------------------
# FIXED TIMESTAMP COLUMN (based on prior validation)
# ------------------------------------------------------------
timestamp_col = "time_since_previously_displayed_frame"

# Fallback (if needed for portability)
if timestamp_col not in df_benign.columns:
    timestamp_col = "inter_arrival_time"

if timestamp_col not in df_benign.columns:
    raise ValueError("No valid timestamp column found in dataset.")

### Timestamp Reconstruction and Label Assignment

In [3]:
### === STEP 2: TIMESTAMP RECONSTRUCTION + LABEL ASSIGNMENT (10 ms bins) ===

# -- Sort and clean time deltas
df_benign = df_benign.sort_index().copy()
df_benign["time_since_previously_displayed_frame"] = (
    df_benign["time_since_previously_displayed_frame"]
    .clip(lower=0)
    .fillna(0)
)

# -- Reconstruct timeline
start_time = pd.Timestamp("2024-01-01 00:00:00")
df_benign["time_elapsed_s"] = df_benign["time_since_previously_displayed_frame"].cumsum()
df_benign["Timestamp_parsed"] = start_time + pd.to_timedelta(df_benign["time_elapsed_s"], unit="s")

# -- Label assignment
df_benign["Label"] = 0

# -- Bin timestamps into 10 ms intervals
df_benign["time_bin_10ms"] = df_benign["Timestamp_parsed"].dt.floor("10ms")

## Aggregation by 10 ms bins and VMR Diagnostics

In [4]:
# --- Begin aggregation pipeline ---

# Bin timestamps
df_benign["time_bin_10ms"] = df_benign["Timestamp_parsed"].dt.floor("10ms")

# Infer protocol intelligently
df_benign = infer_protocol_column(df_benign)

# Prepare numeric columns safely
for col in ["Total Length of Fwd Packet", "Total Length of Bwd Packet", "Flow Duration", "SYN Flag Count"]:
    if col not in df_benign.columns:
        df_benign[col] = 0

df_benign["total_bytes"] = df_benign["Total Length of Fwd Packet"] + df_benign["Total Length of Bwd Packet"]
df_benign["flow_duration"] = df_benign["Flow Duration"]
df_benign["syn_flag"] = df_benign["SYN Flag Count"]

# Aggregations per 10 ms bin
agg_df_10ms = (
    df_benign.groupby("time_bin_10ms").agg(
        event_count=("Protocol", "size"),
        TCP=("Protocol", lambda x: np.sum(x == "TCP")),
        UDP=("Protocol", lambda x: np.sum(x == "UDP")),
        ICMP=("Protocol", lambda x: np.sum(x == "ICMP")),
        OTHER=("Protocol", lambda x: np.sum(x == "OTHER")),
        total_bytes_sum=("total_bytes", "sum"),
        avg_flow_dur=("flow_duration", "mean"),
        syn_flag_sum=("syn_flag", "sum")
    ).reset_index()
)

# Fill NaNs, sort, and finalize
agg_df_10ms = agg_df_10ms.fillna(0).sort_values("time_bin_10ms").reset_index(drop=True)

### Native IAT / Jitter Micro-feature Integration

In [5]:
# === STEP 2B-BENIGN: Native IAT / Jitter Micro-feature Integration ===

# Run for BENIGN
agg_df_10ms = compute_jitter_iat_features(df_benign, agg_df_10ms)
# --- SAFETY PATCH: ensure 'time_bin_10ms' still exists ---
if "time_bin_10ms" not in agg_df_10ms.columns:
    possible_time_cols = [c for c in agg_df_10ms.columns if "time_bin" in c.lower() or "timestamp" in c.lower()]
    if possible_time_cols:
        agg_df_10ms.rename(columns={possible_time_cols[0]: "time_bin_10ms"}, inplace=True)
        print(f"[FIX] Renamed {possible_time_cols[0]} → time_bin_10ms")
    else:
        print("[WARN] time_bin_10ms column missing — reconstructing from index.")
        agg_df_10ms["time_bin_10ms"] = df_benign["Timestamp_parsed"].dt.floor("10ms")

### VMR Diagnostic (10 ms Bins)

In [6]:
# --- Phase 1 → Step 6: Poisson Diagnostic (10 ms bins) ---
poisson_results_benign_10ms = poisson_diagnostic(
    agg_df_10ms,
    label="Benign (10 ms bins)",
    verbose=True
)


=== Poisson Behaviour Diagnostic — Benign (10 ms bins) ===
        Feature  Mean Variance   VMR          Status
    event_count 2.424    8.601 3.549  Over-dispersed
      iat_count 2.424    8.601 3.549  Over-dispersed
            TCP 1.985    9.062 4.565  Over-dispersed
            UDP 0.332    0.399 1.201  Over-dispersed
           ICMP 0.016    0.016 0.997       ≈ Poisson
          OTHER 0.091    0.096 1.054       ≈ Poisson
total_bytes_sum 0.000    0.000 0.000 Under-dispersed
   avg_flow_dur 0.000    0.000 0.000 Under-dispersed
   syn_flag_sum 0.000    0.000 0.000 Under-dispersed

[SUMMARY]
  Over-dispersed : 4 features
  Under-dispersed: 3 features
  ≈ Poisson      : 2 features

[AVERAGE VMR] 1.657


### Temporal Dispersion Integration Feature Integration

In [7]:
### === STEP 4: TEMPORAL-DISPERSION FEATURE INTEGRATION ===

df_disp = agg_df_10ms.copy().sort_values("time_bin_10ms").reset_index(drop=True)

# Ensure time continuity
df_disp["time_bin_10ms"] = pd.to_datetime(df_disp["time_bin_10ms"])
df_disp = df_disp.set_index("time_bin_10ms")

# Choose core signal for λ and dispersion estimation
core_col = "event_count"   # can switch to TCP if focusing on protocol-specific activity

# Rolling λ (expected count per bin)
df_disp["rolling_lambda"] = (
    df_disp[core_col]
    .rolling(window=50, min_periods=1)
    .mean()
)

# Rolling dispersion index (variance / mean over short horizon)
roll_mean = df_disp[core_col].rolling(window=20, min_periods=1).mean()
roll_var  = df_disp[core_col].rolling(window=20, min_periods=1).var()
df_disp["rolling_dispersion"] = np.where(roll_mean > 0, roll_var / roll_mean, 0)

# Stabilize with log transforms
df_disp["log_lambda"] = np.log1p(df_disp["rolling_lambda"])
df_disp["log_dispersion"] = np.log1p(df_disp["rolling_dispersion"])

# Clean output
df_disp = df_disp.fillna(0).reset_index()

### Jitter Feature Integration (Benign)

In [8]:
# === STEP 4A: JITTER FEATURE INTEGRATION (Benign) ===

# Absolute change in event_count between consecutive bins
df_disp["jitter_abs"] = df_disp["event_count"].diff().abs().fillna(0)

# Log-transform for numerical stability
df_disp["log_jitter"] = np.log1p(df_disp["jitter_abs"])


### Feature Preparation

In [9]:
# Select features for training
# Includes count-based and temporal-dispersion features.
feature_cols = [
    # core counts
    "event_count", "TCP", "UDP", "ICMP", "OTHER",
    "syn_flag_sum", "total_bytes_sum",

    # temporal dispersion & macro-jitter
    "log_lambda", "log_dispersion", "jitter_abs", "log_jitter",

    # native micro-IAT jitter features
    "log_iat_mean", "log_iat_std", "log_iat_cv", "log_iat_madd"
]
feature_cols = [c for c in feature_cols if c in df_disp.columns]

# Target variable
target_col = "event_count"

# === STEP 4: Selective Feature Re-Scaling (Robust for Low-Variance Features) ===

# Identify columns with small variance (protocol counts and flags)
minor_cols = [c for c in ["TCP", "UDP", "ICMP", "OTHER", "syn_flag_sum"] if c in df_disp.columns]

if minor_cols:
    scaler_minor = RobustScaler()
    df_disp[minor_cols] = scaler_minor.fit_transform(df_disp[minor_cols])
else:
    print("[WARN] No minor-variance columns found for re-scaling.")

# Build feature matrix and target vector
X_all = df_disp[feature_cols].fillna(0).astype(float).values
y_all = df_disp[target_col].astype(float).values

# Scale features — keep event counts in comparable magnitude
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

### Benign Train - Validation Split (10 ms)

In [10]:
# Chronological split (80% train / 20% validation)
n = len(X_scaled)
split_idx = int(n * 0.8)
X_train, X_val = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_val = y_all[:split_idx], y_all[split_idx:]

### Train & Validate Poisson Model (Benign 10 MS)

In [11]:
### === STEP 6: TRAIN & VALIDATE POISSON MODEL (BENIGN 10 ms) ===

# Initialize model (temporal-aware but stable)
model = ImprovedPoissonConcentrationML(
    model_type="linear",
    q=2,
    confidence_level=0.95,
    lambda_mode="input_based",
    use_conditional=False,
    local_lambda_estimation=True,
    kappa_min=0.1,
    empirical_calibration=True,
    verbose=False
)

# Safe fit on benign training subset
model.safe_fit(X_train, y_train)

# Protect trained model for downstream supplemntal experiments
poisson_model_fitted = model

# save benign λ for re-use (so later attack evaluation does NOT overwrite it)
lambda_benign = model.lambda_estimates.copy()

# Validate on benign hold-out
preds_val, ci_val = model.predict_with_confidence(X_val)
lower_val, upper_val = ci_val[:, 0], ci_val[:, 1]
observed_val = y_val

# inverted or negative intervals if any
inverted = lower_val > upper_val
if np.any(inverted):
    lower_val[inverted], upper_val[inverted] = upper_val[inverted], lower_val[inverted]
    print(f"[WARN] {np.sum(inverted)} intervals inverted — fixed.")

# Compute validation metrics
fp = np.sum((observed_val < lower_val) | (observed_val > upper_val))
tn = len(observed_val) - fp
fp_rate = fp / len(observed_val)

mean_width = np.mean(upper_val - lower_val)
coverage = 1 - fp_rate

print(f"\n[VALIDATION] Benign Validation Results:")
print(f"  False Positives (FP): {fp:,}")
print(f"  True Negatives (TN):  {tn:,}")
print(f"  FP Rate: {fp_rate:.4f}  |  Coverage: {coverage:.4f}")
print(f"  Mean CI Width: {mean_width:.4f}")

# Summary dictionary (for later comparison)
validation_summary = {
    "False Positives": fp,
    "True Negatives": tn,
    "FP Rate": fp_rate,
    "Coverage": coverage,
    "Mean CI Width": mean_width
}


[VALIDATION] Benign Validation Results:
  False Positives (FP): 23,356
  True Negatives (TN):  1,004
  FP Rate: 0.9588  |  Coverage: 0.0412
  Mean CI Width: 0.0130


# PHASE 1(b) Implementation

### Load and Inspect Attack File

In [12]:
# MASTER LOOP — FULL ATTACK PIPELINE

BASE_PATH = "../data"

ATTACK_FILES = {
    "DoS-TCP": os.path.join(BASE_PATH, "DoS-TCP_Flood.csv"),
    "DoS-UDP": os.path.join(BASE_PATH, "DoS-UDP_Flood.csv"),
    "DDoS-TCP": os.path.join(BASE_PATH, "DDoS-TCP_Flood.csv"),
    "DDoS-ICMP": os.path.join(BASE_PATH, "DDoS-ICMP_Fragmentation.csv"),
    "Mirai": os.path.join(BASE_PATH, "Mirai-greip_flood.csv"),
    "Recon": os.path.join(BASE_PATH, "VulnerabilityScan.csv"),
    "WebBased": os.path.join(BASE_PATH, "SqlInjection.csv"),
    "BruteForce": os.path.join(BASE_PATH, "DictionaryBruteForce.csv"),
}
# === GLOBAL STORAGE (FIX FOR MULTI-ATTACK COMBINED STREAM) ===
global_X_attack = []
global_y_attack = []

# Storage (IMPORTANT for later evaluation)
all_attack_results = []
all_attack_dfs = []   # store processed aggregated frames if needed

for attack_label, atk_path in ATTACK_FILES.items():

    # ------------------------------------------------------------
    # BLOCK 1 — LOAD DATA
    # ------------------------------------------------------------
    df_atk = pd.read_csv(atk_path, low_memory=False)

    # --- attack name ---
    attack_name = attack_label

    # ------------------------------------------------------------
    # BLOCK 2 — TIMESTAMP RECONSTRUCTION + LABELING (10 ms)
    # ------------------------------------------------------------
    
    # --- Sort and clean ---
    df_atk = df_atk.sort_index().copy()

    df_atk["time_since_previously_displayed_frame"] = (
        df_atk["time_since_previously_displayed_frame"]
        .clip(lower=0)
        .fillna(0)
    )

    # --- Reconstruct timeline ---
    start_time = pd.Timestamp("2024-01-01 00:00:00")

    df_atk["time_elapsed_s"] = df_atk["time_since_previously_displayed_frame"].cumsum()

    df_atk["Timestamp_parsed"] = start_time + pd.to_timedelta(
        df_atk["time_elapsed_s"], unit="s"
    )

    # --- Label assignment ---
    df_atk["Label"] = 1

    # --- 10 ms binning ---
    df_atk["time_bin_10ms"] = df_atk["Timestamp_parsed"].dt.floor("10ms")


    # ------------------------------------------------------------
    # BLOCK 3 — AGGREGATION (10 ms, COUNT-BASED)
    # ------------------------------------------------------------

    # --- Ensure protocol exists ---
    df_atk = infer_protocol_column(df_atk)

    # --- Safe numeric columns ---
    for col in [
        "Total Length of Fwd Packet",
        "Total Length of Bwd Packet",
        "Flow Duration",
        "SYN Flag Count"
    ]:
        if col not in df_atk.columns:
            df_atk[col] = 0

    df_atk["total_bytes"] = (
        df_atk["Total Length of Fwd Packet"]
        + df_atk["Total Length of Bwd Packet"]
    )

    df_atk["flow_duration"] = df_atk["Flow Duration"]
    df_atk["syn_flag"] = df_atk["SYN Flag Count"]

    # --- Group by 10 ms bin ---
    agg_df_atk_10ms = (
        df_atk.groupby("time_bin_10ms").agg(
            event_count=("Protocol", "size"),
            TCP=("Protocol", lambda x: np.sum(x == "TCP")),
            UDP=("Protocol", lambda x: np.sum(x == "UDP")),
            ICMP=("Protocol", lambda x: np.sum(x == "ICMP")),
            OTHER=("Protocol", lambda x: np.sum(x == "OTHER")),
            total_bytes_sum=("total_bytes", "sum"),
            avg_flow_dur=("flow_duration", "mean"),
            syn_flag_sum=("syn_flag", "sum")
        )
        .reset_index()
    )

    # --- Clean + sort ---
    agg_df_atk_10ms = (
        agg_df_atk_10ms
        .fillna(0)
        .sort_values("time_bin_10ms")
        .reset_index(drop=True)
    )

    # --- Attach attack name ---
    agg_df_atk_10ms["attack_name"] = attack_name


    # ------------------------------------------------------------
    # BLOCK 3.5 — VMR DIAGNOSTICS (Poisson validity check)
    # ------------------------------------------------------------
    poisson_results_attack_10ms = poisson_diagnostic(
        agg_df_atk_10ms,
        label=f"{attack_name} (10 ms bins)",
        verbose=False
    )

    # ------------------------------------------------------------
    # BLOCK 4 — JITTER / IAT FEATURES (Tier 3 critical features)
    # ------------------------------------------------------------

    # Apply your existing function
    agg_df_atk_10ms = compute_jitter_iat_features(df_atk, agg_df_atk_10ms)

    # --- Safety check for time column ---
    if "time_bin_10ms" not in agg_df_atk_10ms.columns:
        raise ValueError("time_bin_10ms column missing after jitter feature computation (attack).")


    # ------------------------------------------------------------
    # BLOCK 5 — TEMPORAL DISPERSION + LOG FEATURES (FIXED)
    # ------------------------------------------------------------

    # --- Rolling lambda ---
    agg_df_atk_10ms["rolling_lambda"] = (
        agg_df_atk_10ms["event_count"]
        .rolling(window=50, min_periods=1)
        .mean()
    )

    # --- Rolling variance ---
    rolling_var = (
        agg_df_atk_10ms["event_count"]
        .rolling(window=50, min_periods=1)
        .var()
    )

    # --- Dispersion ---
    agg_df_atk_10ms["rolling_dispersion"] = (
        rolling_var / (agg_df_atk_10ms["rolling_lambda"] + 1e-10)
    )

    # --- FIX NaNs (CRITICAL) ---
    agg_df_atk_10ms["rolling_dispersion"] = agg_df_atk_10ms["rolling_dispersion"].fillna(0)

    # --- Log transforms ---
    for col in ["event_count", "TCP", "UDP", "ICMP"]:
        agg_df_atk_10ms[f"log_{col}"] = np.log1p(agg_df_atk_10ms[col])

    agg_df_atk_10ms["log_lambda"] = np.log1p(agg_df_atk_10ms["rolling_lambda"])
    agg_df_atk_10ms["log_dispersion"] = np.log1p(agg_df_atk_10ms["rolling_dispersion"])


    # ------------------------------------------------------------
    # BLOCK 5.5 — JITTER FEATURES (RESTORE MISSING TIER3 FEATURES)
    # ------------------------------------------------------------

    # Compute jitter from event_count (same as original logic)
    agg_df_atk_10ms["jitter_abs"] = (
        agg_df_atk_10ms["event_count"]
        .diff()
        .abs()
        .fillna(0)
    )

    agg_df_atk_10ms["log_jitter"] = np.log1p(agg_df_atk_10ms["jitter_abs"])


    # ------------------------------------------------------------
    # BLOCK 6 — FEATURE MATRIX + SCALING (TIER 3 EXACT)
    # ------------------------------------------------------------

    # --- EXACT Tier3 feature list ---
    feature_cols = [
        "event_count", "TCP", "UDP", "ICMP", "OTHER",
        "syn_flag_sum", "total_bytes_sum",
        "log_lambda", "log_dispersion",
        "jitter_abs", "log_jitter",
        "log_iat_mean", "log_iat_std", "log_iat_cv", "log_iat_madd"
    ]

    # Keep only available columns (safety)
    feature_cols = [c for c in feature_cols if c in agg_df_atk_10ms.columns]

    # --- TARGET (same as Tier3) ---
    target_col = "event_count"

    # --- Build matrices ---
    X_attack_all = agg_df_atk_10ms[feature_cols].fillna(0).astype(float).values
    y_attack_all = agg_df_atk_10ms[target_col].astype(float).values

    # ------------------------------------------------------------
    # APPLY SAME SCALING AS BENIGN PIPELINE
    # ------------------------------------------------------------

    # Minor columns scaling (if used in original pipeline)
    minor_cols = [c for c in ["TCP", "UDP", "ICMP", "OTHER", "syn_flag_sum"] if c in agg_df_atk_10ms.columns]

    if len(minor_cols) > 0:
        agg_df_atk_10ms[minor_cols] = scaler_minor.transform(agg_df_atk_10ms[minor_cols])

    # Apply main scaler (VERY IMPORTANT)
    X_attack_scaled = scaler.transform(X_attack_all)

    # === STORE THIS ATTACK FOR GLOBAL COMBINATION ===
    global_X_attack.append(X_attack_scaled)
    global_y_attack.append(y_attack_all)

### COMBINED STREAM

In [13]:
# ============================================================
# BUILD GLOBAL COMBINED STREAM (ALL ATTACKS)
# ============================================================
# --- Stack ALL attacks ---
X_attack_all = np.vstack(global_X_attack)
y_attack_all = np.concatenate(global_y_attack)

# --- Combine with benign ---
X_combined_scaled = np.vstack([X_val, X_attack_all])
y_combined = np.concatenate([y_val, y_attack_all])

combined_labels = np.concatenate([
    np.zeros(len(X_val)),
    np.ones(len(y_attack_all))
])

### ROC ANALYSIS ON ORIGINAL TIER3 PIPELINE (15 FEATURES)

In [14]:
# ============================================================
# ROC ANALYSIS ON ORIGINAL TIER3 PIPELINE (15 FEATURES)
# ============================================================

# Align training data with ROC expectations
X_train_scaled = scaler.transform(X_train)

TARGET_FPRS = [0.001, 0.005, 0.01, 0.02, 0.05]

print('=' * 80)
print('ROC ANALYSIS — PRIMARY DATASET PIPELINE (15 FEATURES)')
print('=' * 80)

# Print kappa
raw_k = getattr(model, 'raw_kappa_', 'N/A')
print(f'[INFO] Raw kappa: {raw_k}')
print(f'[INFO] Clipped kappa: {model.calibration_factor:.6f}')
print(f'[INFO] Features: {X_train_scaled.shape[1]}')
print(f'[INFO] Combined stream: {len(combined_labels)} samples '
      f'({int(np.sum(combined_labels==0))} benign, '
      f'{int(np.sum(combined_labels==1))} attack)')

all_method_results = {}

# ------------------------------------------------------------
# 1. Poisson-CI (NORMALIZED SCORE)
# ------------------------------------------------------------
preds_p, ci_p = model.predict_with_confidence(X_combined_scaled)
lower = ci_p[:, 0]
upper = ci_p[:, 1]

center = (upper + lower) / 2.0
half_width = (upper - lower) / 2.0 + 1e-10

scores_poisson = np.abs(y_combined - center) / half_width

roc_auc, fprs = evaluate_at_fprs(combined_labels, scores_poisson, TARGET_FPRS)
all_method_results['Poisson-CI'] = (roc_auc, fprs)
print(f'[OK] Poisson-CI (normalized): AUC = {roc_auc:.4f}')


# ------------------------------------------------------------
# 2. Poisson-CI (ABSOLUTE RESIDUAL)
# ------------------------------------------------------------
preds_raw = model.predict(X_combined_scaled)
scores_absresid = np.abs(y_combined - preds_raw)

roc_auc2, fprs2 = evaluate_at_fprs(combined_labels, scores_absresid, TARGET_FPRS)
all_method_results['Poisson-CI-AbsResid'] = (roc_auc2, fprs2)
print(f'[OK] Poisson-CI (abs residual): AUC = {roc_auc2:.4f}')


# ------------------------------------------------------------
# 3. Isolation Forest
# ------------------------------------------------------------

iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso.fit(X_train_scaled)

scores_iso = -iso.decision_function(X_combined_scaled)

roc_auc, fprs = evaluate_at_fprs(combined_labels, scores_iso, TARGET_FPRS)
all_method_results['IsoForest'] = (roc_auc, fprs)
print(f'[OK] IsoForest: AUC = {roc_auc:.4f}')


# ------------------------------------------------------------
# 4. One-Class SVM
# ------------------------------------------------------------
n_sub = min(3000, len(X_train_scaled))
idx_sub = np.random.choice(len(X_train_scaled), n_sub, replace=False)

ocsvm = OneClassSVM(kernel='rbf', nu=0.05)
ocsvm.fit(X_train_scaled[idx_sub])

scores_ocsvm = -ocsvm.decision_function(X_combined_scaled)

roc_auc, fprs = evaluate_at_fprs(combined_labels, scores_ocsvm, TARGET_FPRS)
all_method_results['OC-SVM'] = (roc_auc, fprs)
print(f'[OK] OC-SVM: AUC = {roc_auc:.4f}')


# ------------------------------------------------------------
# PRINT RESULTS TABLE
# ------------------------------------------------------------
print(f'\n{"":-<90}')
print(f'{"Method":<25s} {"AUC":>7s}', end='')
for tfpr in TARGET_FPRS:
    print(f'  F1@{tfpr:<6.3f}', end='')
print()

print('-' * 90)

for name, (roc_auc, fprs) in all_method_results.items():
    print(f'{name:<25s} {roc_auc:>7.4f}', end='')
    for tfpr in TARGET_FPRS:
        print(f'  {fprs[tfpr]["F1"]:>8.4f}', end='')
    print()

print(f'\n{"":-<90}')
print('DETAILED VIEW AT FPR = 0.01:')
print(f'{"Method":<25s} {"AUC":>7s} {"Recall":>8s} {"Prec":>8s} {"F1":>8s} {"ActFPR":>10s}')
print('-' * 75)

for name, (roc_auc, fprs) in all_method_results.items():
    m = fprs[0.01]
    print(f'{name:<25s} {roc_auc:>7.4f} {m["Recall"]:>8.4f} {m["Prec"]:>8.4f} {m["F1"]:>8.4f} {m["ActualFPR"]:>10.6f}')

ROC ANALYSIS — PRIMARY DATASET PIPELINE (15 FEATURES)
[INFO] Raw kappa: 1.2823928390461835e-05
[INFO] Clipped kappa: 0.100000
[INFO] Features: 15
[INFO] Combined stream: 261857 samples (24360 benign, 237497 attack)


[OK] Poisson-CI (normalized): AUC = 0.9648
[OK] Poisson-CI (abs residual): AUC = 1.0000


[OK] IsoForest: AUC = 0.5080


[OK] OC-SVM: AUC = 0.7173

------------------------------------------------------------------------------------------
Method                        AUC  F1@0.001   F1@0.005   F1@0.010   F1@0.020   F1@0.050 
------------------------------------------------------------------------------------------
Poisson-CI                 0.9648    0.9643    0.9684    0.9708    0.9726    0.9730
Poisson-CI-AbsResid        1.0000    0.9999    0.9997    0.9995    0.9989    0.9974
IsoForest                  0.5080    0.1283    0.1474    0.1554    0.1721    0.1970
OC-SVM                     0.7173    0.1531    0.1688    0.1826    0.2157    0.3230

------------------------------------------------------------------------------------------
DETAILED VIEW AT FPR = 0.01:
Method                        AUC   Recall     Prec       F1     ActFPR
---------------------------------------------------------------------------
Poisson-CI                 0.9648   0.9442   0.9989   0.9708   0.010057
Poisson-CI-AbsResid      